# Making cutout image using ra, dec of SN and host galaxy (only host galaxy 1) 

Uncomment it if you need to install reporject 

In [ ]:
import sys
!uv pip install --python {sys.executable} reproject 

In [1]:
import pandas as pd

In [2]:
hosts = pd.read_parquet("hosts_SN_Gold.parquet")

In [3]:
hosts

,oid,ra,dec,tier,sn_score,veto_reason,alerce_class,p_SN,p_AGN,p_bogus,...,host_2_absmag_mean,host_2_absmag_std,host_2_absmag_info,host_2_offset_mean,host_2_offset_std,host_2_offset_info,host_2_offset_posterior,any_posterior,none_posterior,has_host
0,170019696272736349,61.497111,-49.245787,Gold,0.984709,NaN,SN,0.984013,0.003316,0.001138,...,-20.778245,1.051610,r,81.805508,0.096498,,0.000000,1.0,3.092591e-14,True
1,170019696299475118,60.150586,-48.419644,Gold,0.982057,NaN,SN,0.939849,0.010425,0.002386,...,-20.657945,1.107231,r,250.845123,0.070689,,0.000000,1.0,2.302329e-14,True
2,170019696438935593,62.952948,-47.749221,Gold,0.980949,NaN,SN,0.961168,0.015453,0.002722,...,-19.739167,1.209000,r,201.583977,0.072736,,0.000000,1.0,6.509423e-14,True
3,170019695992242795,52.823855,-29.691604,Gold,0.971111,NaN,SN,0.991934,0.002390,0.000844,...,-19.956060,1.193836,r,136.482521,0.086874,,0.000000,1.0,2.148735e-14,True
4,170019716316790837,151.548578,1.453601,Gold,0.964962,NaN,SN,0.984521,0.004931,0.001654,...,-20.630996,1.170798,r,136.923751,0.099497,,0.000000,1.0,1.287437e-14,True
5,170019696466198671,61.851091,-47.356485,Gold,0.961308,NaN,SN,0.920890,0.045441,0.003711,...,-19.007349,1.252223,r,282.413503,0.092712,,0.000000,1.0,4.130453e-14,True
6,170019696266444924,59.899046,-50.459673,Gold,0.955944,NaN,SN,0.954770,0.017181,0.003200,...,-19.320000,1.158432,r,90.546307,0.089220,,0.000000,1.0,3.976782e-14,True
7,170019716335140929,149.857915,3.521091,Gold,0.950771,NaN,SN,0.862286,0.046839,0.005344,...,-20.538664,1.196447,r,130.813069,0.101956,,0.000000,1.0,4.080260e-14,True
8,170019696345088120,58.086446,-48.019421,Gold,0.948847,NaN,SN,0.883001,0.058105,0.004422,...,-20.037097,1.193622,r,83.375767,0.101202,,0.000000,1.0,3.151090e-14,True
9,170019696046768167,54.237795,-27.670759,Gold,0.941865,NaN,SN,0.973086,0.008885,0.002719,...,-20.876082,1.174830,r,115.540057,0.095633,,0.000000,1.0,2.104880e-14,True


In [4]:
def imaging_survey(row):
    catalog = str(row["best_cat"]).lower()
    if catalog == "none":
        return "skymapper" if float(row["dec"]) < -30 else "panstarrs"
    return catalog

## 1. Making r-band cutout

In [5]:
from io import BytesIO, StringIO

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from astropy.visualization import ZScaleInterval


def get_survey_cutout(ra, dec, survey, band="r", size_arcsec=120):
    if pd.isna(survey):
        return None
    survey = survey.lower()

    if survey == "decals":
        # The Legacy Survey cutout service limits images to 512 pixels.
        size_px = min(512, round(size_arcsec / 0.262))
        pixscale = max(0.262, size_arcsec / size_px)

        response = requests.get(
            "https://www.legacysurvey.org/viewer/fits-cutout",
            params={
                "ra": ra, "dec": dec,
                "layer": "ls-dr9-south",  # DECaLS imaging used with DR9
                "bands": band,
                "size": size_px,
                "pixscale": pixscale,
            },
            timeout=60,
        )

    elif survey == "panstarrs":
        # First ask MAST which image contains this position.
        response = requests.get(
            "https://ps1images.stsci.edu/cgi-bin/ps1filenames.py",
            params={"ra": ra, "dec": dec, "filters": band},
            timeout=60,
        )
        response.raise_for_status()
        images = pd.read_csv(StringIO(response.text), sep=r"\s+")
        if images.empty:
            raise ValueError(f"No Pan-STARRS {band}-band image at {ra}, {dec}")

        response = requests.get(
            "https://ps1images.stsci.edu/cgi-bin/fitscut.cgi",
            params={
                "ra": ra, "dec": dec,
                "size": round(size_arcsec / 0.25),
                "format": "fits",
                "red": images.iloc[0]["filename"],
            },
            timeout=60,
        )

    elif survey == "skymapper":
        # SkyMapper returns a table of matching image cutouts.
        response = requests.get(
            "https://api.skymapper.nci.org.au/public/siap/dr4/query",
            params={
                "POS": f"{ra},{dec}",
                "SIZE": size_arcsec / 3600,
                "BAND": band,
                "FORMAT": "image/fits",
                "INTERSECT": "CENTER",
                "RESPONSEFORMAT": "CSV",
            },
            timeout=60,
        )
        response.raise_for_status()
        images = pd.read_csv(StringIO(response.text))
        if images.empty:
            raise ValueError(f"No SkyMapper {band}-band image at {ra}, {dec}")

        response = requests.get(images.iloc[0]["get_fits"], timeout=60)

    else:
        raise ValueError(
            "Choose 'decals', 'panstarrs', or 'skymapper'. "
            "GLADE+ has no dedicated image cutout."
        )

    response.raise_for_status()

    with fits.open(BytesIO(response.content)) as hdul:
        image_hdu = next(
            hdu for hdu in hdul
            if hdu.data is not None and hdu.data.ndim == 2
        )
        data = image_hdu.data.astype(float)
        wcs = WCS(image_hdu.header).celestial

    return data, wcs


def plot_prost_host_survey(row, survey, band="r", size_arcsec=120):
    ra, dec = float(row["ra"]), float(row["dec"])
    data, wcs = get_survey_cutout(ra, dec, survey, band, size_arcsec)

    if survey is None:
        return None

    finite = data[np.isfinite(data)]
    if finite.size == 0:
        raise ValueError("The returned cutout has no finite image pixels")
    vmin, vmax = ZScaleInterval().get_limits(finite)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(data, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)

    def mark(ra_value, dec_value, label, marker, color, size):
        if pd.notna(ra_value) and pd.notna(dec_value):
            x, y = wcs.world_to_pixel(
                SkyCoord(float(ra_value), float(dec_value), unit="deg")
            )
            ax.plot(
                x, y, marker=marker, linestyle="none",
                color=color, markersize=size, mew=2,
                markerfacecolor="none" if marker == "o" else color,
                label=label,
            )

    mark(ra, dec, "Transient", "+", "cyan", 8)
    mark(row["host_ra"], row["host_dec"], "PROST host", "o", "red", 8)
    #mark(row["host_2_ra"], row["host_2_dec"], "PROST host 2", "x", "orange", 8)

    ax.set(
        xlim=(0, data.shape[1]),
        ylim=(0, data.shape[0]),
        xlabel="Cutout x (pixels)",
        ylabel="Cutout y (pixels)",
        title=f'{row["tns_name"]} — {survey} {band}-band',
    )
    ax.legend()
    return fig

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

output_dir = Path("./cutout_image_rband")
output_dir.mkdir(parents=True, exist_ok=True)

for _, row in hosts.iterrows():
    survey = imaging_survey(row)

    fig = plot_prost_host_survey(row, survey=survey)

    fig.savefig(
        output_dir / f"prost_host_{row['oid']}_{survey}.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.close(fig)

## 2. Making rgb composite cutout (optional)
But some of case, if they have bad va

In [ ]:
from astropy.visualization import make_lupton_rgb
from reproject import reproject_interp


def plot_prost_host_rgb(row, survey, size_arcsec=120, Q=8, stretch=0.5):
    if pd.isna(survey):
        return None

    survey = survey.lower()
    ra, dec = float(row["ra"]), float(row["dec"])

    # Ordered from blue to red
    bands = ("g", "r", "z") if survey == "decals" else ("g", "r", "i")

    cutouts = {
        band: get_survey_cutout(
            ra, dec, survey, band=band, size_arcsec=size_arcsec
        )
        for band in bands
    }

    # Align every image to the r-band pixel grid.
    reference, wcs = cutouts["r"]
    shape = reference.shape
    aligned = {}
    valid = np.ones(shape, dtype=bool)

    for band in bands:
        data, band_wcs = cutouts[band]

        if band == "r":
            image = data.copy()
            footprint = np.ones(shape)
        else:
            image, footprint = reproject_interp(
                (data, band_wcs),
                wcs,
                shape_out=shape,
            )

        good = np.isfinite(image) & (footprint > 0)
        if not good.any():
            raise ValueError(f"No valid pixels in {band}-band cutout")

        valid &= good

        # Background subtraction and per-band scaling for visualization.
        background = np.median(image[good])
        image = image - background
        scale = np.percentile(image[good], 99.5)

        if scale <= 0:
            raise ValueError(f"No usable signal in {band}-band cutout")

        aligned[band] = np.where(
            good, np.clip(image / scale, 0, None), 0
        )

    rgb = make_lupton_rgb(
        aligned[bands[2]],  # Red: i or z
        aligned["r"],      # Green: r
        aligned["g"],      # Blue: g
        Q=Q,
        stretch=stretch,
    )
    rgb[~valid] = 0

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(rgb, origin="lower")

    def mark(ra_key, dec_key, label, marker, color, size):
        ra_value = row.get(ra_key, np.nan)
        dec_value = row.get(dec_key, np.nan)

        if pd.notna(ra_value) and pd.notna(dec_value):
            coord = SkyCoord(
                float(ra_value), float(dec_value), unit="deg"
            )
            x, y = wcs.world_to_pixel(coord)
            ax.plot(
                x, y,
                marker=marker,
                linestyle="none",
                color=color,
                markersize=size,
                markeredgewidth=1.5,
                markerfacecolor="none",
                label=label,
            )

    mark("ra", "dec", "Transient", "+", "cyan", 12)
    mark("host_ra", "host_dec", "PROST host", "o", "red", 12)

    ax.set(
        xlim=(-0.5, shape[1] - 0.5),
        ylim=(-0.5, shape[0] - 0.5),
        xlabel="Cutout x (pixels)",
        ylabel="Cutout y (pixels)",
        title=f'{row["oid"]} — {survey} {"".join(bands)}',
    )
    ax.legend()
    fig.tight_layout()
    return fig

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

output_dir = Path("./cutout_image_color_composite")
output_dir.mkdir(parents=True, exist_ok=True)

for _, row in hosts.iterrows():
    survey = imaging_survey(row)
    print(survey)

    fig = plot_prost_host_rgb(row, survey, size_arcsec=120)

    fig.savefig(
        output_dir / f"prost_host_{row['oid']}_{survey}.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.close(fig)